<a href="https://colab.research.google.com/github/CaginAyhanOzden/python-task-manager/blob/main/task_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Task Manager

This project is a console-based task management application developed in Python.  
It allows users to add, list, edit, delete, search, filter, sort, and complete tasks.  
Tasks are saved in a text file so that the data can be loaded again when the program is restarted.

In [ ]:
from datetime import datetime

try:
    from google.colab import output
    COLAB_MODE = True
except ImportError:
    import os
    COLAB_MODE = False


FILE_NAME = "tasks.txt"


# ---------------------------------------------------------
# Screen and basic helper functions
# ---------------------------------------------------------

def clear_screen():
    """Clears the output area after each completed operation."""
    if COLAB_MODE:
        output.clear()
    else:
        os.system("cls" if os.name == "nt" else "clear")


def pause_and_clear():
    """Lets the user read the result before returning to the main menu."""
    input("\nPress Enter to return to the main menu...")
    clear_screen()


def create_task(title, priority="Medium", status="Not Started", due_date="No Due Date"):
    """Creates a task using a standard dictionary structure."""
    return {
        "title": title,
        "priority": priority,
        "status": status,
        "due_date": due_date
    }


def remove_numbering(line):
    """
    Removes the task number at the beginning of a saved line.
    Example: '1. Python Project | ...' becomes 'Python Project | ...'
    """
    parts = line.split(". ", 1)

    if len(parts) == 2 and parts[0].isdigit():
        return parts[1]

    return line


# ---------------------------------------------------------
# Validation functions
# ---------------------------------------------------------

def is_valid_due_date(due_date):
    """Checks whether the due date is valid or intentionally left empty."""
    if due_date == "No Due Date":
        return True

    try:
        datetime.strptime(due_date, "%d-%m-%Y")
        return True
    except ValueError:
        return False


def is_valid_title(title):
    """
    Checks whether the task title can be safely saved.
    The '|' character is not allowed because it is used as a separator in the file.
    """
    if title.strip() == "":
        print("Task title cannot be empty.")
        return False

    if "|" in title:
        print("Task title cannot contain the '|' character.")
        return False

    return True


# ---------------------------------------------------------
# File operations
# ---------------------------------------------------------

def load_tasks():
    """
    Loads tasks from the text file.
    Invalid or corrupted lines are skipped instead of stopping the whole program.
    """
    tasks = []

    try:
        with open(FILE_NAME, "r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()

                if line == "":
                    continue

                line = remove_numbering(line)
                parts = [part.strip() for part in line.split("|")]

                if len(parts) != 4:
                    print("Invalid task format found. This line was skipped.")
                    continue

                title = parts[0]
                priority_part = parts[1]
                status_part = parts[2]
                due_date_part = parts[3]

                if not priority_part.startswith("Priority: "):
                    print("Invalid priority format found. This line was skipped.")
                    continue

                if not status_part.startswith("Status: "):
                    print("Invalid status format found. This line was skipped.")
                    continue

                if not due_date_part.startswith("Due Date: "):
                    print("Invalid due date format found. This line was skipped.")
                    continue

                priority = priority_part.replace("Priority: ", "", 1)
                status = status_part.replace("Status: ", "", 1)
                due_date = due_date_part.replace("Due Date: ", "", 1)

                if priority not in ["High", "Medium", "Low"]:
                    print("Invalid priority value found. This line was skipped.")
                    continue

                if status not in ["Not Started", "In Progress", "Completed"]:
                    print("Invalid status value found. This line was skipped.")
                    continue

                if not is_valid_due_date(due_date):
                    print("Invalid due date value found. This line was skipped.")
                    continue

                tasks.append(create_task(title, priority, status, due_date))

        print("Tasks loaded successfully.")
        return tasks

    except FileNotFoundError:
        print("Task file not found. A new task list has been created.")
        return []

    except Exception:
        print("An error occurred while loading tasks. A new task list has been created.")
        return []


def save_tasks(tasks):
    """Saves all tasks into a readable text file format."""
    try:
        with open(FILE_NAME, "w", encoding="utf-8") as file:
            for index, task in enumerate(tasks, start=1):
                file.write(
                    f"{index}. {task['title']} | "
                    f"Priority: {task['priority']} | "
                    f"Status: {task['status']} | "
                    f"Due Date: {task['due_date']}\n"
                )

        print("Tasks saved successfully.")

    except Exception:
        print("An error occurred while saving tasks.")


# ---------------------------------------------------------
# Display functions
# ---------------------------------------------------------

def display_task(task, index):
    """Displays a single task in a clean and readable format."""
    print(f"{index}. {task['title']}")
    print(f"   Priority : {task['priority']}")
    print(f"   Status   : {task['status']}")
    print(f"   Due Date : {task['due_date']}")


def list_tasks(tasks):
    """Lists all saved tasks."""
    if len(tasks) == 0:
        print("No tasks found.")
    else:
        print("\n--- TASK LIST ---")

        for index, task in enumerate(tasks, start=1):
            display_task(task, index)


def show_menu():
    """Displays the main menu."""
    print("\n--- SMART TASK MANAGER ---")
    print("1. List Tasks")
    print("2. Add New Task")
    print("3. Edit Task")
    print("4. Delete Task")
    print("5. Mark Task as Completed")
    print("6. Search Tasks")
    print("7. Filter Tasks")
    print("8. Sort Tasks")
    print("9. Show Statistics")
    print("0. Exit")


# ---------------------------------------------------------
# Input selection functions
# ---------------------------------------------------------

def get_priority():
    """Asks the user to choose a valid priority level."""
    while True:
        print("\nPriority Options:")
        print("1. High")
        print("2. Medium")
        print("3. Low")

        choice = input("Choose priority (1-3): ").strip()

        if choice == "1":
            return "High"
        elif choice == "2":
            return "Medium"
        elif choice == "3":
            return "Low"
        else:
            print("Invalid priority choice. Please enter 1, 2, or 3.")


def get_status(allow_default=False):
    """
    Asks the user to choose a valid status.
    While adding a new task, pressing Enter sets the status to 'Not Started'.
    """
    while True:
        print("\nStatus Options:")
        print("1. Not Started")
        print("2. In Progress")
        print("3. Completed")

        if allow_default:
            choice = input("Choose status (1-3) or press Enter for Not Started: ").strip()
        else:
            choice = input("Choose status (1-3): ").strip()

        if allow_default and choice == "":
            return "Not Started"

        if choice == "1":
            return "Not Started"
        elif choice == "2":
            return "In Progress"
        elif choice == "3":
            return "Completed"
        else:
            print("Invalid status choice. Please enter 1, 2, or 3.")


def get_due_date():
    """Asks the user for a due date and validates the DD-MM-YYYY format."""
    while True:
        due_date = input("Enter due date (DD-MM-YYYY) or press Enter to skip: ").strip()

        if due_date == "":
            return "No Due Date"

        try:
            datetime.strptime(due_date, "%d-%m-%Y")
            return due_date

        except ValueError:
            print("Invalid date format. Please use DD-MM-YYYY.")


def get_task_number(tasks, action):
    """Gets a valid task number for edit, delete, or completion operations."""
    try:
        task_number = int(input(f"Enter the task number to {action}: "))

        if task_number < 1 or task_number > len(tasks):
            print("Invalid task number.")
            return None

        return task_number

    except ValueError:
        print("Please enter a valid number.")
        return None


# ---------------------------------------------------------
# Main task operations
# ---------------------------------------------------------

def add_task(tasks):
    """Adds a new task with title, priority, status, and due date."""
    title = input("Enter a new task title: ").strip()

    if not is_valid_title(title):
        return

    priority = get_priority()
    status = get_status(allow_default=True)
    due_date = get_due_date()

    new_task = create_task(title, priority, status, due_date)
    tasks.append(new_task)

    print("Task added successfully.")
    save_tasks(tasks)


def edit_task(tasks):
    """Edits an existing task without forcing the user to change every field."""
    list_tasks(tasks)

    if len(tasks) == 0:
        return

    task_number = get_task_number(tasks, "edit")

    if task_number is None:
        return

    task = tasks[task_number - 1]

    print("\nPress Enter to keep the current value.")

    new_title = input(f"New title ({task['title']}): ").strip()

    if new_title != "":
        if not is_valid_title(new_title):
            return
        task["title"] = new_title

    change_priority = input("Do you want to change priority? (y/n): ").strip().lower()

    if change_priority == "y":
        task["priority"] = get_priority()

    change_status = input("Do you want to change status? (y/n): ").strip().lower()

    if change_status == "y":
        task["status"] = get_status()

    change_due_date = input("Do you want to change due date? (y/n): ").strip().lower()

    if change_due_date == "y":
        task["due_date"] = get_due_date()

    print("Task updated successfully.")
    save_tasks(tasks)


def delete_task(tasks):
    """Deletes a selected task by its task number."""
    list_tasks(tasks)

    if len(tasks) == 0:
        return

    task_number = get_task_number(tasks, "delete")

    if task_number is None:
        return

    deleted_task = tasks.pop(task_number - 1)
    print(f"'{deleted_task['title']}' deleted successfully.")
    save_tasks(tasks)


def mark_task_completed(tasks):
    """Marks a selected task as completed."""
    list_tasks(tasks)

    if len(tasks) == 0:
        return

    task_number = get_task_number(tasks, "mark as completed")

    if task_number is None:
        return

    tasks[task_number - 1]["status"] = "Completed"

    print("Task marked as completed successfully.")
    save_tasks(tasks)


# ---------------------------------------------------------
# Search, filter, sort, and statistics
# ---------------------------------------------------------

def search_tasks(tasks):
    """Searches tasks by title, priority, status, or due date."""
    if len(tasks) == 0:
        print("No tasks found.")
        return

    keyword = input("Enter keyword to search: ").strip().lower()

    if keyword == "":
        print("Search keyword cannot be empty.")
        return

    results = []

    for task in tasks:
        if (
            keyword in task["title"].lower()
            or keyword in task["priority"].lower()
            or keyword in task["status"].lower()
            or keyword in task["due_date"].lower()
        ):
            results.append(task)

    if len(results) == 0:
        print("No matching tasks found.")
    else:
        print("\n--- SEARCH RESULTS ---")

        for index, task in enumerate(results, start=1):
            display_task(task, index)


def filter_tasks(tasks):
    """Filters tasks by priority, status, due date, or missing due date."""
    if len(tasks) == 0:
        print("No tasks found.")
        return

    print("\nFilter Options:")
    print("1. Filter by Priority")
    print("2. Filter by Status")
    print("3. Filter by Due Date")
    print("4. Show Tasks Without Due Date")

    choice = input("Choose filter option (1-4): ").strip()
    results = []

    if choice == "1":
        selected_priority = get_priority()
        results = [task for task in tasks if task["priority"] == selected_priority]

    elif choice == "2":
        selected_status = get_status()
        results = [task for task in tasks if task["status"] == selected_status]

    elif choice == "3":
        selected_due_date = get_due_date()
        results = [task for task in tasks if task["due_date"] == selected_due_date]

    elif choice == "4":
        results = [task for task in tasks if task["due_date"] == "No Due Date"]

    else:
        print("Invalid filter option.")
        return

    if len(results) == 0:
        print("No tasks matched the selected filter.")
    else:
        print("\n--- FILTER RESULTS ---")

        for index, task in enumerate(results, start=1):
            display_task(task, index)


def get_due_date_for_sort(task):
    """Converts due date text into a sortable date value."""
    if task["due_date"] == "No Due Date":
        return datetime.max

    return datetime.strptime(task["due_date"], "%d-%m-%Y")


def sort_tasks(tasks):
    """Sorts tasks by priority, status, due date, or title."""
    if len(tasks) == 0:
        print("No tasks found.")
        return

    print("\nSort Options:")
    print("1. Sort by Priority")
    print("2. Sort by Status")
    print("3. Sort by Due Date")
    print("4. Sort by Title")

    choice = input("Choose sort option (1-4): ").strip()

    if choice == "1":
        priority_order = {
            "High": 1,
            "Medium": 2,
            "Low": 3
        }

        tasks.sort(key=lambda task: priority_order[task["priority"]])
        print("Tasks sorted by priority.")

    elif choice == "2":
        status_order = {
            "Not Started": 1,
            "In Progress": 2,
            "Completed": 3
        }

        tasks.sort(key=lambda task: status_order[task["status"]])
        print("Tasks sorted by status.")

    elif choice == "3":
        tasks.sort(key=get_due_date_for_sort)
        print("Tasks sorted by due date.")

    elif choice == "4":
        tasks.sort(key=lambda task: task["title"].lower())
        print("Tasks sorted by title.")

    else:
        print("Invalid sort option.")
        return

    save_tasks(tasks)
    list_tasks(tasks)


def show_statistics(tasks):
    """Shows a short summary of all task data."""
    total_tasks = len(tasks)

    if total_tasks == 0:
        print("No tasks found.")
        return

    completed_count = 0
    in_progress_count = 0
    not_started_count = 0

    high_count = 0
    medium_count = 0
    low_count = 0

    no_due_date_count = 0

    for task in tasks:
        if task["status"] == "Completed":
            completed_count += 1
        elif task["status"] == "In Progress":
            in_progress_count += 1
        elif task["status"] == "Not Started":
            not_started_count += 1

        if task["priority"] == "High":
            high_count += 1
        elif task["priority"] == "Medium":
            medium_count += 1
        elif task["priority"] == "Low":
            low_count += 1

        if task["due_date"] == "No Due Date":
            no_due_date_count += 1

    completion_rate = (completed_count / total_tasks) * 100

    print("\n--- TASK STATISTICS ---")
    print(f"Total Tasks       : {total_tasks}")
    print(f"Completed         : {completed_count}")
    print(f"In Progress       : {in_progress_count}")
    print(f"Not Started       : {not_started_count}")
    print(f"High Priority     : {high_count}")
    print(f"Medium Priority   : {medium_count}")
    print(f"Low Priority      : {low_count}")
    print(f"No Due Date       : {no_due_date_count}")
    print(f"Completion Rate   : {completion_rate:.2f}%")


# ---------------------------------------------------------
# Program loop
# ---------------------------------------------------------

def main():
    """Runs the Smart Task Manager application."""
    tasks = load_tasks()

    while True:
        show_menu()
        choice = input("Your choice (0-9): ").strip()

        if choice == "1":
            list_tasks(tasks)
            pause_and_clear()

        elif choice == "2":
            add_task(tasks)
            pause_and_clear()

        elif choice == "3":
            edit_task(tasks)
            pause_and_clear()

        elif choice == "4":
            delete_task(tasks)
            pause_and_clear()

        elif choice == "5":
            mark_task_completed(tasks)
            pause_and_clear()

        elif choice == "6":
            search_tasks(tasks)
            pause_and_clear()

        elif choice == "7":
            filter_tasks(tasks)
            pause_and_clear()

        elif choice == "8":
            sort_tasks(tasks)
            pause_and_clear()

        elif choice == "9":
            show_statistics(tasks)
            pause_and_clear()

        elif choice == "0":
            print("Exiting the program...")
            break

        else:
            print("Invalid choice! Please enter a number between 0 and 9.")
            pause_and_clear()


if __name__ == "__main__":
    main()